In [1]:
from database.MongoDBClient import MongoDBClient
from database.Recipe import Recipe
from transformers import AutoTokenizer
from model.model import Chefformer
from config.ModelSettings import ModelSettings
import torch

/Users/michalekb/Documents/GitHub_Personal/chefformer_repo/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/michalekb/Documents/GitHub_Personal/chefformer_repo/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ModelSettings()

ModelSettings(name='Chefformer', vocab_size=50257, embedding_size=768, max_context_length=512, num_attn_heads=1, dropout_prob=0.1)

In [3]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
chefformer = Chefformer(tokenizer)


In [4]:
s = """Title: Italian Cream Cake

Ingredients:
- Buttermilk (1 cup)
- Baking soda (1 teaspoon)
- Butter (1/2 cup)
- Shortening (1/2 cup)
- White sugar (2 cups)
- Eggs (5)
- Vanilla extract (1 teaspoon)
- Flaked coconut (1 cup)
- Baking powder (1 teaspoon)
- All-purpose flour (2 cups)
- Cream cheese (8 ounces)
- Butter (1/2 cup)
- Vanilla extract (1 teaspoon)
- Confectioners' sugar (4 cups)
- Light cream (2 tablespoons)
- Chopped walnuts (1/2 cup)
- Sweetened flaked coconut (1 cup)
...

Instructions: Finely grate the zest from 4 of the lemons, then pulse in a mini food processor with 3 tablespoons salt. Squeeze the juice from 2 of the zested lemons into a large bowl; add the lemongrass, olive oil, Sriracha, garlic, fish sauce, brown sugar and 1 teaspoon of the lemon salt. Thinly slice the 2 remaining lemons and add to the bowl.Using kitchen shears, cut through the shrimp shells along the outer curve, leaving the shells on. Remove the veins and rinse the shrimp. Add to the bowl with the lemongrass marinade and toss; cover and refrigerate for 1 hour. Meanwhile, soak 12 to 15 wooden skewers in water.Preheat a grill to medium high. Thread the shrimp onto the skewers. Brush the grill with vegetable oil, then add the lemon slices and grill until they begin to char, turning once, about 2 minutes; transfer to a platter. Stir the mint into the remaining marinade. Add the shrimp skewers to the grill and cook, brushing with the marinade-mint mixture, until the shells begin to char, 2 to 3 minutes per side. Transfer to the platter and sprinkle with the lemon salt to taste."""


s = 'hey how are you'

s = ['hey how', 'are you doing today']

In [5]:
x = tokenizer(s, return_tensors='pt', padding=True, truncation=True, max_length=512)
x['input_ids']

tensor([[20342,   703, 50256, 50256],
        [  533,   345,  1804,  1909]])

In [6]:
embeddings = chefformer.get_embeddings(s)
print(embeddings)
print(embeddings.shape)

tensor([[[ 1.1263e+00, -3.7298e-01,  2.0400e+00,  ..., -1.1903e+00,
          -1.3069e+00, -1.6908e+00],
         [-1.7139e+00, -4.6099e+00,  1.2651e+00,  ..., -2.1076e+00,
           1.7072e+00,  3.0926e+00],
         [-3.2298e-01,  1.0643e+00, -1.3823e+00,  ..., -2.1667e-01,
           6.5757e-01, -1.2915e+00],
         [-5.1150e-01, -2.2339e-01, -3.2948e+00,  ..., -1.0174e-02,
          -2.0327e+00, -1.9188e+00]],

        [[ 4.7159e-01, -1.3716e+00,  1.7949e+00,  ..., -8.1115e-01,
          -4.4954e+00,  1.5325e-01],
         [-2.1268e+00, -4.7109e+00,  3.2286e-01,  ...,  2.1948e-01,
           2.5459e+00,  2.2296e+00],
         [-1.9697e-01,  3.8387e-01, -2.1948e-01,  ...,  1.7155e-03,
           2.1277e+00, -1.0521e+00],
         [-3.7838e-01, -2.4267e+00, -1.4673e+00,  ..., -3.1694e-01,
          -7.6439e-01, -1.8160e+00]]], grad_fn=<AddBackward0>)
torch.Size([2, 4, 768])


In [7]:
# Test parts of attention

def scaled_dot_product_attention(query, key, value):
    batch_size, seq_length, d_attn_head = query.shape[0], query.shape[1], query.shape[-1]
    attenion_weights = torch.bmm(query, torch.transpose(key, 1, 2)) # Q * K.T... (batch_size, seq_len, seq_len)
    attenion_weights = attenion_weights / torch.sqrt(torch.tensor(d_attn_head, dtype=torch.float32)) # Scale the scores

    mask = torch.tril(torch.ones(batch_size, seq_length, seq_length, dtype=torch.int)) # Gather attention matrix... (batch_size, seq_len, seq_len)
    attenion_weights[mask == 0] = float('-inf') # Set scores to -inf where tokens are masked

    attenion_weights = torch.softmax(attenion_weights, dim=-1) # Take softmax along rows (each token's row should sum to 1)
    return torch.bmm(attenion_weights, value)


example_embeddings = torch.tensor([[[0.5, 0.5, 0.5],
                           [-0.1, -0.1, -0.1]],

                           [[0.3, 0.3, 0.3],
                            [1.0, 1.0, 1.0]]])

print(scaled_dot_product_attention(example_embeddings, example_embeddings, example_embeddings))
print()


attn_output = chefformer.test_attention_head(['hey how are you', 'today'])
print(attn_output)
print(attn_output.shape)

tensor([[[0.5000, 0.5000, 0.5000],
         [0.1844, 0.1844, 0.1844]],

        [[0.3000, 0.3000, 0.3000],
         [0.8395, 0.8395, 0.8395]]])

tensor([[[-0.4364, -0.4219,  0.8880,  ..., -0.8760, -0.3322, -0.6555],
         [-0.6758, -0.4586,  0.7391,  ..., -0.5425, -0.5061,  0.1460],
         [-0.6696, -0.3069, -0.0277,  ..., -0.4053, -0.0367,  0.5821],
         [-0.3122, -0.4798, -0.4054,  ..., -0.5454,  0.3041,  0.1448]],

        [[ 0.3622,  0.8441, -0.3967,  ...,  0.7061, -0.5748, -0.5581],
         [-0.0041, -0.3086,  0.3199,  ...,  0.5852, -0.9211,  0.5417],
         [-0.2356, -0.2861,  0.0339,  ...,  0.5323, -0.9135,  0.2552],
         [ 0.1290, -0.5377,  0.2263,  ...,  0.5116, -0.8701,  0.4886]]],
       grad_fn=<BmmBackward0>)
torch.Size([2, 4, 768])


In [8]:
# Test multi head attention
attn_output = chefformer.test_multihead_masked_self_attention(['hey how are you', 'today'])
print(attn_output)
print(attn_output.shape)

tensor([[[-0.5419,  0.2686,  0.6823,  ..., -0.4366,  0.2828,  0.0234],
         [-0.3868,  0.2210,  0.5527,  ..., -0.2852,  0.2921, -0.0547],
         [ 0.1597,  0.0194,  0.0514,  ...,  0.1820,  0.2904, -0.3145],
         [-0.3560,  0.1675,  0.2212,  ..., -0.0959,  0.0963, -0.0875]],

        [[-1.0637,  0.2471,  0.1113,  ..., -0.1094, -0.1902, -0.5181],
         [ 0.4033,  0.2585,  0.3006,  ...,  0.3861,  0.2996,  0.0760],
         [ 0.3268, -0.0703,  0.0850,  ...,  0.3605,  0.1156,  0.1954],
         [ 0.0381, -0.1762,  0.1411,  ...,  0.2123, -0.0256,  0.2334]]],
       grad_fn=<ViewBackward0>)
torch.Size([2, 4, 768])


In [9]:
# Test feed foward network
ff_output = chefformer.test_feed_forward(['hey how are you', 'today'])
print(ff_output)
print(ff_output.shape)

tensor([[[ 0.0095,  0.0224,  0.0707,  ..., -0.0330,  0.0419, -0.0628],
         [ 0.0022,  0.0264,  0.0479,  ..., -0.0382,  0.0406, -0.0328],
         [-0.0574, -0.0761, -0.0619,  ..., -0.0777,  0.0528,  0.0465],
         [-0.0298, -0.0905,  0.0985,  ..., -0.0141,  0.0019, -0.0039]],

        [[ 0.1136, -0.1366,  0.1155,  ..., -0.1119,  0.0328,  0.0659],
         [ 0.0609, -0.0018, -0.0072,  ..., -0.0361,  0.0744,  0.0519],
         [ 0.0106, -0.0591, -0.0341,  ...,  0.0457,  0.0486, -0.0004],
         [-0.0055, -0.0190,  0.0146,  ...,  0.0705,  0.0732, -0.0325]]],
       grad_fn=<ViewBackward0>)
torch.Size([2, 4, 768])
